In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.compose import TransformedTargetRegressor

In [2]:
df = pd.read_csv('HDB_Resale_Enriched.csv')

In [3]:
print("1. Starting data preparation...")

# Target variable: resale_price_per_sqm
df['resale_price_per_sqm'] = df['resale_price'] / df['floor_area_sqm']

# Precise Time Features
df['month_dt'] = pd.to_datetime(df['month'])
df = df.sort_values('month_dt').reset_index(drop=True) # Sort chronologically

df['transaction_year'] = df['month_dt'].dt.year

# Create a continuous index (Month 1, Month 2...) to capture linear market trends
df['time_index'] = (df['transaction_year'] - df['transaction_year'].min()) * 12 + df['month_dt'].dt.month


# Map storey_range to mid-point and rename to 'storey_mid' based on your new list
def get_storey_midpoint(val):
    try:
        parts = val.split(' TO ')
        return (int(parts[0]) + int(parts[1])) / 2
    except:
        return np.nan
df['storey_mid'] = df['storey_range'].apply(get_storey_midpoint)

# Parse remaining_lease to a continuous numeric value (Years)
def parse_lease(val):
    if pd.isna(val): return np.nan
    val_str = str(val).lower()
    years, months = 0, 0
    year_match = re.search(r'(\d+)\s*year', val_str)
    month_match = re.search(r'(\d+)\s*month', val_str)
    
    if year_match: years = int(year_match.group(1))
    elif val_str.isdigit(): years = int(val_str)
    if month_match: months = int(month_match.group(1))
    return years + (months / 12.0)
df['remaining_lease_years'] = df['remaining_lease'].apply(parse_lease)

# Structural Event Flags (based on proposal feedback)
# 1. Post-COVID impact (Circuit breaker started Apr 2020, WFH caused space demand)
df['is_post_covid'] = (df['month'] >= '2020-04').astype(int)
# 2. Cooling Measure: Dec 2021 (ABSD rates raised)
df['cooling_measure_dec2021'] = (df['month'] >= '2021-12').astype(int)
# 3. Cooling Measure: Sep 2022 (15-month wait-out period for private downgraders)
df['cooling_measure_sep2022'] = (df['month'] >= '2022-09').astype(int)
# 4. Interest Rate Hikes: US Fed started aggressive hikes around mid-2022
df['high_interest_rates'] = (df['month'] >= '2022-07').astype(int)

structural_flags = [
    'is_post_covid', 'cooling_measure_dec2021', 
    'cooling_measure_sep2022', 'high_interest_rates'
]

# ---------------------------------------------------------
# Feature Definitions & One-Hot Encoding
# ---------------------------------------------------------
# Your defined continuous features
base_features = [
    'floor_area_sqm', 'storey_mid', 'remaining_lease_years', 
    'nearest_mrt_1_dist_m', 'nearest_mrt_2_dist_m', 'nearest_mrt_3_dist_m',
    'nearest_school_dist_m', 'hdb_latitude', 'hdb_longitude', 
    'transaction_year', 'time_index'
]

# categorical features
cat_features = ['town', 'flat_type', 'flat_model']

print("Performing One-Hot Encoding for categorical features...")
# Convert categorical text into 0/1 columns. drop_first=True prevents multi-collinearity
df_encoded = pd.get_dummies(df, columns=cat_features, drop_first=True, dtype=int)

# Extract the newly created one-hot encoded column names
encoded_cat_cols = [col for col in df_encoded.columns if any(col.startswith(f"{cat}_") for cat in cat_features)]

# Combine all features into final training list
final_features = base_features + structural_flags + encoded_cat_cols
target = 'resale_price_per_sqm'

# Drop rows with missing values
data = df_encoded[final_features + [target]].dropna()
print(f"Data preparation complete. Total features used: {len(final_features)}")
print(f"Total rows ready for modeling: {len(data):,}")

# ---------------------------------------------------------
# Splitting & Training
# ---------------------------------------------------------
print("\n2. Performing Chronological Splitting...")
train_df = data[data['transaction_year'] <= 2023]
val_df = data[data['transaction_year'] == 2024]
test_df = data[data['transaction_year'] >= 2025]

X_train, y_train = train_df[final_features], train_df[target]
X_val, y_val = val_df[final_features], val_df[target]
X_test, y_test = test_df[final_features], test_df[target]

print(f"Training set (<=2023): {len(train_df):,} rows")
print(f"Validation set (2024): {len(val_df):,} rows")
print(f"Test set (>=2025): {len(test_df):,} rows")

1. Starting data preparation...
Performing One-Hot Encoding for categorical features...
Data preparation complete. Total features used: 66
Total rows ready for modeling: 224,665

2. Performing Chronological Splitting...
Training set (<=2023): 169,150 rows
Validation set (2024): 27,832 rows
Test set (>=2025): 27,683 rows


In [4]:
print("\n=======================================================")
print("STARTING GRID SEARCH FOR INDIVIDUAL MODELS")
print("=======================================================")
# ---------------------------------------------------------
# Setting up Grid Search CV
# ---------------------------------------------------------
# 1. Setup Evaluation Metrics
def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Custom scorer for GridSearchCV
grid_scorer = make_scorer(rmse_scorer, greater_is_better=False)

def evaluate_best_model(model, X, y_true):
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    return rmse, mae, r2

# 2. Define the Base Models
base_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    # Lasso max_iter increased to ensure it converges
    "Lasso": Lasso(max_iter=10000), 
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

# 3. Define the Hyperparameter Grids
# Note: 'regressor__' is required to wrap models in TransformedTargetRegressor
param_grids = {
    "Linear Regression": {}, # No hyperparameters to tune
    "Ridge": {
        "regressor__alpha": [0.1, 1.0, 10.0, 100.0]
    },
    "Lasso": {
        # Using very small alphas to prevent Lasso from turning all features to 0
        "regressor__alpha": [0.0001, 0.001, 0.01, 0.1] 
    },
    "Decision Tree": {
        "regressor__max_depth": [10, 15, 20, None],
        "regressor__min_samples_split": [2, 10, 20]
    },
    "Random Forest": {
        "regressor__n_estimators": [50, 100],
        "regressor__max_depth": [10, 15, 20]
    },
    "Gradient Boosting": {
        "regressor__n_estimators": [50, 100],
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__max_depth": [3, 5, 7]
    }
}

print("Initializing 5-Fold Time-Series Cross Validation...")
tscv = TimeSeriesSplit(n_splits=5)

results = []
best_trained_models = {} # save the best models to use in Stacking

for name in base_models.keys():
    print(f"\n--- Tuning {name} ---")
    
    # Wrap model with Log-Transformer
    log_model = TransformedTargetRegressor(
        regressor=base_models[name], 
        func=np.log, 
        inverse_func=np.exp
    )
    
    # Setup Grid Search
    grid_search = GridSearchCV(
        estimator=log_model,
        param_grid=param_grids[name],
        cv=tscv,
        scoring=grid_scorer,
        n_jobs=-1,
        verbose=1
    )
    
    # Run Grid Search on 2017-2023 Training Data
    grid_search.fit(X_train, y_train)
    
    # Extract Best Model
    best_model = grid_search.best_estimator_
    best_trained_models[name] = best_model
    
    # Print the winning parameters (removing the 'regressor__' prefix for readability)
    best_params_clean = {k.replace('regressor__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"Best Params: {best_params_clean}")
    
    # Evaluate on Validation (2024) and Test (2025) Sets
    val_rmse, val_mae, val_r2 = evaluate_best_model(best_model, X_val, y_val)
    test_rmse, test_mae, test_r2 = evaluate_best_model(best_model, X_test, y_test)
    
    results.append({
        "Model": name,
        "Best Params": str(best_params_clean),
        "Val 2024 RMSE": round(val_rmse, 2), "Val 2024 MAE": round(val_mae, 2), "Val 2024 R2": round(val_r2, 4),
        "Test 2025 RMSE": round(test_rmse, 2), "Test 2025 MAE": round(test_mae, 2), "Test 2025 R2": round(test_r2, 4)
    })


STARTING GRID SEARCH FOR INDIVIDUAL MODELS
Initializing 5-Fold Time-Series Cross Validation...

--- Tuning Linear Regression ---
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best Params: {}

--- Tuning Ridge ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Params: {'alpha': 0.1}

--- Tuning Lasso ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Params: {'alpha': 0.0001}

--- Tuning Decision Tree ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'max_depth': None, 'min_samples_split': 20}

--- Tuning Random Forest ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Params: {'max_depth': 20, 'n_estimators': 100}

--- Tuning Gradient Boosting ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100}


In [5]:
print("=======================================================")
print("BUILDING THE OPTIMIZED STACKING ENSEMBLE")
print("=======================================================")
print("Using the best tuned Random Forest and Gradient Boosting as base estimators...")

# Extract the purely tuned regressors
tuned_rf = best_trained_models["Random Forest"].regressor_
tuned_gb = best_trained_models["Gradient Boosting"].regressor_
tuned_ridge = best_trained_models["Ridge"].regressor_

estimators = [
    ('rf', tuned_rf),
    ('gb', tuned_gb),
    ('ridge', tuned_ridge)
]

# Build Stacking Regressor
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, 
    n_jobs=-1
)

# Wrap Stacking in Log-Transformer and Train
log_stacking = TransformedTargetRegressor(regressor=stacking_model, func=np.log, inverse_func=np.exp)
log_stacking.fit(X_train, y_train)

# Evaluate Stacking
val_rmse, val_mae, val_r2 = evaluate_best_model(log_stacking, X_val, y_val)
test_rmse, test_mae, test_r2 = evaluate_best_model(log_stacking, X_test, y_test)

results.append({
    "Model": "Stacking Ensemble",
    "Best Params": "Meta: Ridge | Base: Tuned RF & GB",
    "Val 2024 RMSE": round(val_rmse, 2), "Val 2024 MAE": round(val_mae, 2), "Val 2024 R2": round(val_r2, 4),
    "Test 2025 RMSE": round(test_rmse, 2), "Test 2025 MAE": round(test_mae, 2), "Test 2025 R2": round(test_r2, 4)
})

print("\n=======================================================")
print("ALL TUNING & EVALUATION COMPLETED. FINAL RESULTS:")
print("=======================================================")
results_df = pd.DataFrame(results)

# Display just the metrics for easy reading
display_df = results_df[['Model', 'Test 2025 MAE', 'Test 2025 RMSE', 'Test 2025 R2']]
print(display_df.to_string(index=False))


BUILDING THE OPTIMIZED STACKING ENSEMBLE
Using the best tuned Random Forest and Gradient Boosting as base estimators...

ALL TUNING & EVALUATION COMPLETED. FINAL RESULTS:
                        Model  Test 2025 MAE  Test 2025 RMSE  Test 2025 R2
            Linear Regression         849.92         1020.71        0.6261
                        Ridge         849.83         1020.56        0.6263
                        Lasso         839.79         1013.60        0.6313
                Decision Tree         836.01         1025.20        0.6229
                Random Forest         827.34          985.85        0.6513
            Gradient Boosting         803.08          935.77        0.6858
Stacking Ensemble (Optimized)         725.87          865.13        0.7314
